# Notebook 4 — Preparing the Model for Pretraining
**Adapted for Jupyter Notebook / Kubeflow**

Configures a LLaMA-style transformer, demonstrates three weight initialisation strategies,
and saves the depth-pruned model as `./data/SmolLM2-26L-pruned-init/` for the training loop.

**Learning objectives:**
1. Configure a LLaMA-style transformer with `LlamaConfig`
2. Compare three weight initialisation strategies: random, continued pretraining, depth pruning
3. Observe the difference between random and pretrained model inference
4. Save the initialised model for use in Notebook 5

In [1]:
import warnings
warnings.filterwarnings("ignore")
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "transformers", "torch", "-q"])

CompletedProcess(args=['/home/jovyan/llmall/venv/bin/python', '-m', 'pip', 'install', 'transformers', 'torch', '-q'], returncode=0)

In [2]:
import os, gc
import torch

# ── Data directory — same across all notebooks ────────────────────────
data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)
print(f"Data directory : {data_dir}")

def fix_torch_seed(seed=42):
    """Fix all random seeds for reproducibility."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

fix_torch_seed()
print(f"PyTorch version: {torch.__version__}")
print(f"Device         : {'GPU — ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Data directory : ./data
PyTorch version: 2.5.1+cu124
Device         : GPU — NVIDIA A100-SXM4-80GB


## 1. Model Configuration

We use **LlamaConfig** — the same architecture family as Meta's LLaMA, Mistral, Qwen, and SmolLM2.

> **Config vs weights:** The config is the architectural blueprint (number of layers, hidden size, etc.).
> Weights are the learned values that fill that blueprint.
> You define the config first; weights are added later — either randomly or from a pretrained checkpoint.

In [3]:
from transformers import LlamaConfig

# Inspect the default LlamaConfig (full LLaMA-7B architecture)
config = LlamaConfig()
print("Default LlamaConfig (LLaMA-7B scale):")
print(config)

Default LlamaConfig (LLaMA-7B scale):
LlamaConfig {
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.3",
  "use_cache": true,
  "vocab_size": 32000
}



### Customise for the Demo Model

We match the SmolLM2-360M architecture — a compact, fully open-source LLaMA-compatible model.

| Parameter | LLaMA-7B | Our demo | What it controls |
|---|---|---|---|
| `num_hidden_layers` | 32 | 12 | Transformer blocks (depth) |
| `hidden_size` | 4096 | 960 | Width of each layer |
| `intermediate_size` | 11008 | 2560 | MLP feed-forward width |
| `num_attention_heads` | 32 | 15 | Parallel attention heads |
| `num_key_value_heads` | 32 | 5 | GQA heads — fewer = less memory |
| `torch_dtype` | float32 | bfloat16 | Half-precision — halves memory |
| `use_cache` | True | False | Must be False for gradient checkpointing |

In [4]:
config.num_hidden_layers  = 12      # reduced from 32
config.hidden_size        = 960     # matches SmolLM2-360M
config.intermediate_size  = 2560    # matches SmolLM2-360M MLP width
config.num_attention_heads= 15      # matches SmolLM2-360M
config.num_key_value_heads= 5       # Grouped Query Attention (GQA)
config.torch_dtype        = "bfloat16"
config.use_cache          = False   # required for gradient checkpointing in training
print("✅ Custom config set")
print(config)

✅ Custom config set
LlamaConfig {
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 960,
  "initializer_range": 0.02,
  "intermediate_size": 2560,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 15,
  "num_hidden_layers": 12,
  "num_key_value_heads": 5,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.46.3",
  "use_cache": false,
  "vocab_size": 32000
}



## 2. Weight Initialisation Strategies

### Strategy 2a — Random Initialisation

All weights drawn from a truncated normal distribution (mean=0, std=0.02).
Use when: training a brand-new model from scratch with massive data and compute.

> Inference on a randomly initialised model produces **gibberish** — this demonstrates
> exactly why pretraining on large corpora is necessary.

In [5]:
from transformers import LlamaForCausalLM, AutoTokenizer, TextStreamer

def print_nparams(model):
    n = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {n:,}  (~{n/1e6:.0f}M)")

# Build model with random weights
model = LlamaForCausalLM(config)

# GPU change: move model to GPU, use bfloat16
model = model.to(torch.bfloat16).to(model.device)

print_nparams(model)

Total parameters: 208,920,000  (~209M)


In [6]:
# Inspect the raw random weights
layer_name = "model.layers.0.self_attn.q_proj.weight"
for name, param in model.named_parameters():
    if name == layer_name:
        print(f"First 15 random weights of '{layer_name}':")
        print(param.data.view(-1)[:15])
        break

First 15 random weights of 'model.layers.0.self_attn.q_proj.weight':
tensor([ 0.0181, -0.0469,  0.0078,  0.0284,  0.0124,  0.0349,  0.0325,  0.0029,
        -0.0199, -0.0077,  0.0146,  0.0231,  0.0292, -0.0097, -0.0222],
       dtype=torch.bfloat16)


In [7]:
# ── Inference on random weights — expect complete gibberish ──────────
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")
prompt    = "Large language models are trained on"
inputs    = tokenizer(prompt, return_tensors="pt").to(model.device)

print(f"Prompt: '{prompt}'")
print("\nOutput (random weights — gibberish expected):")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        use_cache=True    # ← True is fine on GPU for random model
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Prompt: 'Large language models are trained on'

Output (random weights — gibberish expected):
Large language models are trained on airplane airplane airplaneSyn airplaneurableSynSynurableurableurableurable iv singersLAN iv iv ivLAN ivLAN ArgumentLAN ivLAN ArgumentLANLANLANLANLANLANICSLANICSLANICSLANICSLANICSLANICS absorbs issue issue issue issue absorbs absorbs absorbs absorbs absorbs conditions absorbs conditions absorbs conditions absorbs conditions absorbs conditions absorbs conditions


In [8]:
del model, outputs
gc.collect()
print("Memory cleared.")

Memory cleared.


### Strategy 2b — Continued Pretraining (Reuse Pretrained Weights)

Load an already-pretrained model and continue training on new domain data.
This is the most compute-efficient strategy — you start from a strong base.

**Real examples:** CodeLlama = LLaMA-2 continued on code. BioMedLM = GPT-2 continued on PubMed.

**Reference model:** `HuggingFaceTB/SmolLM2-360M`
- 360M parameters, LLaMA architecture, fully open-source
- Pretrained on FineWeb-Edu (web text) + The Stack (code)

In [9]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M",
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")
print("✅ Pretrained model loaded:")
print_nparams(model)
print(f"Architecture : {model.config.model_type}")
print(f"Layers       : {model.config.num_hidden_layers}")

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Pretrained model loaded:
Total parameters: 361,821,120  (~362M)
Architecture : llama
Layers       : 32


In [10]:
# ── Inference on pretrained weights — expect coherent text ───────────
prompt = "Large language models are trained on"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print(f"Prompt: '{prompt}'")
print("\nSmolLM2-360M output:")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        use_cache=True
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Prompt: 'Large language models are trained on'

SmolLM2-360M output:
Large language models are trained on large corpora of text, and the training process is time-consuming.

The training process is time-consuming because it requires a large corpus of text.

The training process is time-consuming because it requires a large corpus of text.

The training process is time-consuming because it requires a


In [11]:
del model, outputs
gc.collect()
print("Memory cleared.")

Memory cleared.


### Strategy 2c — Downscaling (Depth Pruning)

Create a smaller model by removing the two innermost middle layers from a pretrained model.

**Why middle layers?**
- Bottom layers: learn low-level syntax and token patterns
- Top layers: learn high-level reasoning and task behaviour
- Middle layers: most redundant — safest to remove

We downscale SmolLM2-360M from **30 layers → 26 layers**.

In [12]:
from transformers import AutoConfig

model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M",
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")

print("Before downscaling:")
print_nparams(model)
print(f"Number of layers: {len(model.model.layers)}")

Before downscaling:
Total parameters: 361,821,120  (~362M)
Number of layers: 32


In [13]:
# Remove the two middle layers
total_layers = len(model.model.layers)
keep_bottom  = total_layers // 2 - 1
keep_top     = total_layers // 2 - 1

layers               = model.model.layers
model.model.layers   = layers[:keep_bottom] + layers[-keep_top:]

config_pruned = AutoConfig.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M",
    num_hidden_layers=len(model.model.layers),
)
model.config = config_pruned

print("After downscaling:")
print_nparams(model)
print(f"Layers remaining : {len(model.model.layers)}  (removed {total_layers - len(model.model.layers)} middle layers)")

After downscaling:
Total parameters: 342,156,480  (~342M)
Layers remaining : 30  (removed 2 middle layers)


In [14]:
# ── Inference on the downscaled model ────────────────────────────────
prompt = "Large language models are trained on"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print(f"Prompt: '{prompt}'")
print("\nDownscaled model output (still coherent, slightly degraded):")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        use_cache=False     
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Prompt: 'Large language models are trained on'

Downscaled model output (still coherent, slightly degraded):
Large language models are trained on a large corpus of text, and the model is trained to predict the next word in the text.

The model is trained on a large corpus of text, and the model is trained to predict the next word in the text.

The model is trained on a large corpus of text, and the model is


## 3. Save the Model

In [15]:
# Save as SmolLM2-26L-pruned-init
# Name breakdown: SmolLM2 (family) | 26L (26 layers) | pruned (depth pruning) | init (not yet trained)

save_path = os.path.join(data_dir, "SmolLM2-26L-pruned-init")
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model saved to: {save_path}")
print("Saved files:")
for f in sorted(os.listdir(save_path)):
    size_kb = os.path.getsize(os.path.join(save_path, f)) / 1024
    print(f"  {f:<45} {size_kb:.1f} KB")

✅ Model saved to: ./data/SmolLM2-26L-pruned-init
Saved files:
  config.json                                   0.8 KB
  generation_config.json                        0.1 KB
  merges.txt                                    455.5 KB
  model.safetensors                             668304.3 KB
  special_tokens_map.json                       0.8 KB
  tokenizer.json                                3440.1 KB
  tokenizer_config.json                         3.6 KB
  vocab.json                                    781.9 KB


In [16]:
del model, outputs
gc.collect()
print("Memory cleared.")

Memory cleared.


## Summary

| Concept | Detail |
|---|---|
| `LlamaConfig` | Defines architecture blueprint independently of weights |
| Random init | Truncated normal — baseline for training from scratch; produces gibberish |
| Continued pretraining | Load pretrained weights and train further on domain data |
| Depth pruning | Remove middle transformer layers to create a smaller model |
| `print_nparams()` | Essential sanity check after any architectural change |
| `bfloat16` | Half-precision — halves memory footprint |
| `use_cache=False` | Required when gradient checkpointing is enabled during training |
| Output | `./data/SmolLM2-26L-pruned-init/` → input to Notebook 5 |